# Structural Pattern Matching

Added in Python 3.10, `match`/`case` goes far beyond a switch statement. It matches the *structure* of data: sequences, mappings, class instances. It can bind variables, apply guards, and handle wildcards in a single expression.

**What's inside:** literal matching, sequence patterns, mapping patterns, class patterns, OR patterns, guards, nested patterns, and a comparison with `isinstance` chains and dispatch dicts.

**Learn more:** [match statements](https://docs.python.org/3/reference/compound_stmts.html#the-match-statement) · [PEP 634](https://peps.python.org/pep-0634/)

## 1. Matching literals

In [1]:
def http_status(code):
    match code:
        case 200:
            return 'OK'
        case 301 | 302:        # OR pattern
            return 'Redirect'
        case 404:
            return 'Not Found'
        case 500:
            return 'Server Error'
        case _:                # wildcard: matches anything
            return f'Unknown ({code})'

for code in [200, 301, 404, 418]:
    print(code, '->', http_status(code))

200 -> OK
301 -> Redirect
404 -> Not Found
418 -> Unknown (418)


## 2. Matching sequences

In [2]:
def describe_point(point):
    match point:
        case [0, 0]:
            return 'origin'
        case [x, 0]:
            return f'on x-axis at {x}'
        case [0, y]:
            return f'on y-axis at {y}'
        case [x, y]:
            return f'point ({x}, {y})'
        case _:
            return 'not a 2D point'

for p in [(0, 0), (3, 0), (0, -2), (4, 5), (1, 2, 3)]:
    print(p, '->', describe_point(p))

(0, 0) -> origin
(3, 0) -> on x-axis at 3
(0, -2) -> on y-axis at -2
(4, 5) -> point (4, 5)
(1, 2, 3) -> not a 2D point


In [3]:
def first_and_rest(seq):
    match seq:
        case []:               # empty
            return 'empty'
        case [only]:
            return f'single: {only}'
        case [first, *rest]:   # *rest captures the tail
            return f'head={first}, tail={rest}'

for s in [[], [42], [1, 2, 3, 4]]:
    print(s, '->', first_and_rest(s))

[] -> empty
[42] -> single: 42
[1, 2, 3, 4] -> head=1, tail=[2, 3, 4]


## 3. Matching mappings

In [4]:
def handle_event(event):
    match event:
        case {'type': 'click', 'button': btn}:
            return f'click: {btn}'
        case {'type': 'keypress', 'key': k, 'shift': True}:
            return f'shift+{k}'
        case {'type': 'keypress', 'key': k}:
            return f'key: {k}'
        case {'type': t}:
            return f'unknown event type: {t}'
        case _:
            return 'not an event'

events = [
    {'type': 'click', 'button': 'left'},
    {'type': 'keypress', 'key': 'a', 'shift': True},
    {'type': 'keypress', 'key': 'enter'},
    {'type': 'scroll', 'delta': 3},
]
for e in events:
    print(handle_event(e))

click: left
shift+a
key: enter
unknown event type: scroll


## 4. Matching class instances

In [5]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

@dataclass
class Circle:
    center: Point
    radius: float

@dataclass
class Rectangle:
    top_left: Point
    bottom_right: Point

def describe_shape(shape):
    match shape:
        case Circle(center=Point(x=0, y=0), radius=r):
            return f'circle of radius {r} centred at origin'
        case Circle(center=c, radius=r):
            return f'circle of radius {r} at ({c.x}, {c.y})'
        case Rectangle(top_left=tl, bottom_right=br):
            w = br.x - tl.x
            h = br.y - tl.y
            return f'rectangle {w}×{h}'

shapes = [
    Circle(Point(0, 0), 5),
    Circle(Point(3, 4), 2),
    Rectangle(Point(0, 4), Point(6, 0)),
]
for s in shapes:
    print(describe_shape(s))

circle of radius 5 centred at origin
circle of radius 2 at (3, 4)
rectangle 6×-4


## 5. Guards: adding conditions with `if`

In [6]:
def classify(value):
    match value:
        case int(n) if n < 0:
            return 'negative int'
        case int(n) if n == 0:
            return 'zero'
        case int(n):
            return f'positive int: {n}'
        case float(f) if f != f:   # NaN check
            return 'NaN'
        case float(f):
            return f'float: {f}'
        case str(s) if s.isupper():
            return f'ALL CAPS: {s}'
        case str(s):
            return f'string: {s}'

for v in [-3, 0, 7, float('nan'), 3.14, 'HELLO', 'world']:
    print(repr(v), '->', classify(v))

-3 -> negative int
0 -> zero
7 -> positive int: 7
nan -> NaN
3.14 -> float: 3.14
'HELLO' -> ALL CAPS: HELLO
'world' -> string: world


## 6. Nested patterns

In [7]:
# JSON-style data, a common real-world use case
def extract_name(data):
    match data:
        case {'user': {'name': str(name), 'active': True}}:
            return f'active user: {name}'
        case {'user': {'name': str(name), 'active': False}}:
            return f'inactive user: {name}'
        case {'error': str(msg)}:
            return f'error: {msg}'
        case _:
            return 'unknown structure'

records = [
    {'user': {'name': 'Alice', 'active': True}},
    {'user': {'name': 'Bob', 'active': False}},
    {'error': 'not found'},
    {'other': 'data'},
]
for r in records:
    print(extract_name(r))

active user: Alice
inactive user: Bob
error: not found
unknown structure


## 7. match vs isinstance chains vs dispatch dicts

In [8]:
# The same logic written three ways
from dataclasses import dataclass

@dataclass
class Add:  left: object; right: object
@dataclass
class Mul:  left: object; right: object
@dataclass
class Num:  value: int

# ── match/case (most readable for structural dispatch) ──
def evaluate_match(expr):
    match expr:
        case Num(value=v):             return v
        case Add(left=l, right=r):    return evaluate_match(l) + evaluate_match(r)
        case Mul(left=l, right=r):    return evaluate_match(l) * evaluate_match(r)

# ── isinstance chain ──
def evaluate_isinstance(expr):
    if isinstance(expr, Num):   return expr.value
    if isinstance(expr, Add):   return evaluate_isinstance(expr.left) + evaluate_isinstance(expr.right)
    if isinstance(expr, Mul):   return evaluate_isinstance(expr.left) * evaluate_isinstance(expr.right)

expr = Add(Mul(Num(2), Num(3)), Num(4))   # (2*3)+4
print(evaluate_match(expr))               # 10
print(evaluate_isinstance(expr))          # 10

10
10
